In [6]:
import polars as pl

annot_path = '../references/human_ref/gencode.v49.annotation.gtf'
only_protein_coding = False

annot_df = pl.scan_csv(
    annot_path, separator='\t', comment_prefix='#', has_header=False,
    new_columns=['chr', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attr']
)

# count transcripts per gene
transcript_query = (
    annot_df
    .select(['feature', 'attr'])
    .filter(pl.col('feature') == 'transcript')
    .filter(pl.col('attr').str.contains('gene_type "protein_coding";') if only_protein_coding else pl.lit(True))
    .with_columns(pl.col('attr').str.extract(r'gene_id "([^"]+)";').alias('gene_id'))
    .select(
        pl.col('gene_id').unique(maintain_order=True),
        pl.col('gene_id').unique_counts().alias('transcript_count')
    )
    .filter(
        pl.col('transcript_count') >= 3,
        pl.col('transcript_count') <= 20
    )
)
sample_gene_pool = transcript_query.collect()
all_genes = sample_gene_pool['gene_id'].to_list()

gene_query = (
    annot_df
    .filter(pl.col('feature') == 'gene')
    .with_columns(pl.col('attr').str.extract(r'gene_id "([^"]+)";').alias('gene_id'))
    .filter(pl.col('gene_id').is_in(all_genes))
    .with_columns((pl.col('end') - pl.col('start') + 1).alias('gene_length'))
)
gene_df = gene_query.collect()
gene_df

chr,source,feature,start,end,score,strand,frame,attr,gene_id,gene_length
str,str,str,i64,i64,str,str,str,str,str,i64
"""chr1""","""HAVANA""","""gene""",28589,31109,""".""","""+""",""".""","""gene_id ""ENSG00000243485.6""; g…","""ENSG00000243485.6""",2521
"""chr1""","""HAVANA""","""gene""",34553,37595,""".""","""-""",""".""","""gene_id ""ENSG00000237613.3""; g…","""ENSG00000237613.3""",3043
"""chr1""","""HAVANA""","""gene""",96638,105742,""".""","""+""",""".""","""gene_id ""ENSG00000308314.1""; g…","""ENSG00000308314.1""",9105
"""chr1""","""HAVANA""","""gene""",257864,359681,""".""","""-""",""".""","""gene_id ""ENSG00000292994.2""; g…","""ENSG00000292994.2""",101818
"""chr1""","""HAVANA""","""gene""",266753,284291,""".""","""+""",""".""","""gene_id ""ENSG00000286448.2""; g…","""ENSG00000286448.2""",17539
…,…,…,…,…,…,…,…,…,…,…
"""chrY""","""HAVANA""","""gene""",56919422,56954169,""".""","""-""",""".""","""gene_id ""ENSG00000310550.1""; g…","""ENSG00000310550.1""",34748
"""chrY""","""HAVANA""","""gene""",57067747,57130289,""".""","""+""",""".""","""gene_id ""ENSG00000292366.2""; g…","""ENSG00000292366.2""",62543
"""chrY""","""HAVANA""","""gene""",57184216,57199537,""".""","""+""",""".""","""gene_id ""ENSG00000292373.3""; g…","""ENSG00000292373.3""",15322


In [2]:
sample_query = (
    annot_df
    .filter(
        pl.col('attr').str.contains_any(sample_gene_id)
    )
)
sample_df = sample_query.collect()
sample_df

chr,source,feature,start,end,score,strand,frame,attr
str,str,str,i64,i64,str,str,str,str
"""chr1""","""HAVANA""","""gene""",157745733,157777269,""".""","""-""",""".""","""gene_id ""ENSG00000132704.17""; …"
"""chr1""","""HAVANA""","""transcript""",157745733,157770310,""".""","""-""",""".""","""gene_id ""ENSG00000132704.17""; …"
"""chr1""","""HAVANA""","""exon""",157767231,157770310,""".""","""-""",""".""","""gene_id ""ENSG00000132704.17""; …"
"""chr1""","""HAVANA""","""exon""",157766855,157766971,""".""","""-""",""".""","""gene_id ""ENSG00000132704.17""; …"
"""chr1""","""HAVANA""","""exon""",157749650,157749677,""".""","""-""",""".""","""gene_id ""ENSG00000132704.17""; …"
…,…,…,…,…,…,…,…,…
"""chrX""","""HAVANA""","""CDS""",74525750,74525893,""".""","""+""","""0""","""gene_id ""ENSG00000147100.12""; …"
"""chrX""","""HAVANA""","""exon""",74531333,74532608,""".""","""+""",""".""","""gene_id ""ENSG00000147100.12""; …"
"""chrX""","""HAVANA""","""CDS""",74531333,74531407,""".""","""+""","""0""","""gene_id ""ENSG00000147100.12""; …"


In [8]:
genes = (
    sample_df
    .filter(pl.col('feature') == 'gene')
    .with_columns((pl.col('end') - pl.col('start')).alias('length'))
)
print(genes.select('length', pl.sum('length').alias('total_length')))

shape: (20, 2)
┌────────┬──────────────┐
│ length ┆ total_length │
│ ---    ┆ ---          │
│ i64    ┆ i64          │
╞════════╪══════════════╡
│ 31536  ┆ 1215405      │
│ 10623  ┆ 1215405      │
│ 28778  ┆ 1215405      │
│ 7370   ┆ 1215405      │
│ 80298  ┆ 1215405      │
│ …      ┆ …            │
│ 25227  ┆ 1215405      │
│ 18838  ┆ 1215405      │
│ 14275  ┆ 1215405      │
│ 6704   ┆ 1215405      │
│ 112535 ┆ 1215405      │
└────────┴──────────────┘
